# 📊 Pengantar Data Science — Pertemuan 7
## Pengantar Machine Learning: Regresi Linear

---

| Info | Detail |
|------|--------|
| **Mata Kuliah** | Pengantar Data Science (200302305) |
| **Program Studi** | Informatika |
| **Pertemuan** | 7 — Pengantar Machine Learning: Regresi Linear |
| **Koordinator** | Syahid Abdullah, S.Si, M.Kom |
| **Nama** | Ranu Ratmaja |
| **NIM** | 230401010104 |

---

### 🎯 Capaian Pembelajaran
1. Menjelaskan perbedaan **Supervised Learning** dan **Unsupervised Learning**
2. Menjelaskan konsep **Regresi Linear**, cost function (MSE), dan gradient descent
3. Membedakan **Simple** dan **Multiple Linear Regression**
4. Mengimplementasikan Regresi Linear menggunakan `LinearRegression` dari scikit-learn
5. Menghitung dan menginterpretasikan metrik evaluasi **MAE, RMSE, dan R²**
6. Membangun pipeline prediksi **end-to-end** dengan visualisasi Actual vs Predicted dan Residual Plot

---
## 📦 0. Import Library

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

import warnings
warnings.filterwarnings('ignore')

plt.rcParams['figure.dpi'] = 120
sns.set_theme(style='whitegrid', palette='muted')

print('✅ Semua library berhasil diimport!')

---
## 🧠 1. Konsep Machine Learning

### 1.1 Apa itu Machine Learning?

> **Machine Learning (ML)** adalah cabang kecerdasan buatan yang memungkinkan komputer belajar dari data tanpa diprogram secara eksplisit.

Definisi Tom Mitchell (1997): *"A computer program is said to learn from experience E with respect to some task T and some performance measure P, if its performance on T, as measured by P, improves with experience E."*

### 1.2 Supervised vs Unsupervised Learning

| Aspek | Supervised Learning | Unsupervised Learning |
|-------|--------------------|-----------------------|
| **Label (y)** | Ada | Tidak ada |
| **Analogi** | Belajar dari soal + kunci jawaban | Belajar mandiri |
| **Task** | Klasifikasi & Regresi | Clustering & Dim. Reduction |
| **Contoh Algoritma** | Linear Regression, SVM, Decision Tree | K-Means, DBSCAN, PCA |

### 1.3 Classification vs Regression

| Aspek | Classification | Regression |
|-------|----------------|------------|
| **Output (y)** | Kelas / label diskret | Nilai **kontinu** |
| **Contoh output** | Spam / Bukan spam | Rp 450.000.000 |
| **Metrik** | Accuracy, F1, Precision | MAE, RMSE, R² |
| **Contoh kasus** | Deteksi fraud | Prediksi gaji, harga rumah |

> 🎯 Pertemuan ini fokus pada **Regression** — khususnya **Regresi Linear**.

---
## 📈 2. Regresi Linear

### 2.1 Konsep dan Intuisi

Regresi Linear mencari **garis lurus terbaik** yang melewati titik-titik data untuk memprediksi nilai baru.

**Kapan digunakan:**
- ✅ Target (y) berupa nilai kontinu
- ✅ Hubungan antara X dan y bersifat **linear**
- ✅ Butuh model yang **interpretable**
- ✅ Sebagai **baseline model** sebelum coba algoritma kompleks

### 2.2 Persamaan Regresi Linear

**Simple Linear Regression (1 variabel):**
$$\hat{y} = \beta_0 + \beta_1 \cdot x$$

**Multiple Linear Regression (banyak variabel):**
$$\hat{y} = \beta_0 + \beta_1 x_1 + \beta_2 x_2 + \cdots + \beta_n x_n$$

| Simbol | Nama | Interpretasi |
|--------|------|-------------|
| $\hat{y}$ | Nilai prediksi | Output yang diprediksi model |
| $\beta_0$ | Intercept | Nilai $\hat{y}$ saat $x = 0$ |
| $\beta_1$ | Slope/Koefisien | Perubahan $\hat{y}$ per unit $x$ |

**Contoh:** $\hat{y} = 3.5 + 2.1x$ → untuk 5 thn pengalaman: $\hat{y} = 3.5 + 2.1 \times 5 = 14.0$ juta rupiah

### 2.3 Cost Function: Mean Squared Error (MSE)

$$e_i = y_i - \hat{y}_i \quad \text{(residual)}$$

$$MSE = \frac{1}{n} \cdot \sum(y_i - \hat{y}_i)^2$$

**Tujuan:** Temukan $\beta_0$ dan $\beta_1$ yang **meminimalkan MSE** → solusi *Ordinary Least Squares (OLS)*

**Mengapa dikuadratkan?**
- Nilai negatif dan positif tidak saling menghapus
- Penalti lebih besar untuk kesalahan lebih besar
- Fungsi *differentiable* → bisa dioptimalkan

### 2.4 Gradient Descent

Algoritma iteratif meminimalkan cost function:
1. Inisialisasi $\beta$ secara acak
2. Hitung gradient (turunan parsial MSE)
3. Update: $\beta := \beta - \alpha \cdot \nabla MSE$
4. Ulangi hingga konvergen

| Learning Rate (α) | Dampak |
|-------------------|--------|
| Terlalu kecil | Stabil tapi sangat lambat |
| Tepat (~0.01) | Konvergen cepat & stabil ✅ |
| Terlalu besar | Overshooting, tidak konvergen ❌ |

> ⚠️ `sklearn.LinearRegression` menggunakan **OLS** (solusi analitik langsung), bukan iterasi Gradient Descent.

---
## 🛠️ 3. Hands-on: Pipeline End-to-End Prediksi Gaji

**Pipeline:** Generate Data → EDA → Preprocessing → Training → Evaluasi → Visualisasi

### Langkah 1 — Generate & Eksplorasi Dataset

In [ ]:
# ── Generate Dataset Sintetis ─────────────────────────────────────
np.random.seed(42)
n = 300

pengalaman = np.random.uniform(0, 20, n)
edu        = np.random.choice([0, 1, 2], n)       # SMA=0, D3=1, S1=2
kota       = np.random.choice(['Jakarta', 'Surabaya', 'Bandung'], n)

gaji = (
    3.0
    + 2.2 * pengalaman
    + 1.5 * edu
    + np.where(kota == 'Jakarta', 4.0, 0)
    + np.random.normal(0, 2, n)
)

df = pd.DataFrame({
    'pengalaman': pengalaman,
    'edu'       : edu,
    'kota'      : kota,
    'gaji'      : gaji
})

print('='*50)
print('INFORMASI DATASET')
print('='*50)
print(f'Jumlah baris   : {df.shape[0]}')
print(f'Jumlah kolom   : {df.shape[1]}')
print(f'Missing values : {df.isnull().sum().sum()}')
print()
df.head(10)

In [ ]:
# ── Statistik Deskriptif ──────────────────────────────────────────
print('Statistik Deskriptif:')
df.describe().round(2)

In [ ]:
# ── Rata-rata Gaji per Kota ───────────────────────────────────────
print('Rata-rata Gaji per Kota:')
print(df.groupby('kota')['gaji'].agg(['mean','std','count']).round(2))

In [ ]:
# ── EDA Visualisasi ───────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle('EDA Dataset Prediksi Gaji Karyawan', fontsize=14, fontweight='bold')

# Plot 1: Scatter pengalaman vs gaji
colors_kota = {'Jakarta': '#E63946', 'Surabaya': '#457B9D', 'Bandung': '#2A9D8F'}
for k, c in colors_kota.items():
    mask = df['kota'] == k
    axes[0].scatter(df.loc[mask,'pengalaman'], df.loc[mask,'gaji'],
                    alpha=0.5, label=k, color=c, s=25)
axes[0].set_xlabel('Pengalaman (tahun)')
axes[0].set_ylabel('Gaji (juta Rp)')
axes[0].set_title('Pengalaman vs Gaji per Kota')
axes[0].legend()

# Plot 2: Boxplot per kota
df.boxplot(column='gaji', by='kota', ax=axes[1],
           boxprops=dict(color='#457B9D'),
           medianprops=dict(color='#E63946', linewidth=2))
axes[1].set_title('Distribusi Gaji per Kota')
axes[1].set_xlabel('Kota')
axes[1].set_ylabel('Gaji (juta Rp)')
plt.sca(axes[1]); plt.title('Distribusi Gaji per Kota')

# Plot 3: Boxplot per pendidikan
edu_map = {0:'SMA', 1:'D3', 2:'S1'}
df['edu_label'] = df['edu'].map(edu_map)
df.boxplot(column='gaji', by='edu_label', ax=axes[2],
           boxprops=dict(color='#2A9D8F'),
           medianprops=dict(color='#E63946', linewidth=2))
axes[2].set_title('Distribusi Gaji per Pendidikan')
axes[2].set_xlabel('Pendidikan')
axes[2].set_ylabel('Gaji (juta Rp)')
plt.sca(axes[2]); plt.title('Distribusi Gaji per Pendidikan')

plt.tight_layout()
plt.show()
df.drop(columns=['edu_label'], inplace=True)

In [ ]:
# ── Heatmap Korelasi ──────────────────────────────────────────────
plt.figure(figsize=(5, 4))
corr = df[['pengalaman','edu','gaji']].corr()
sns.heatmap(corr, annot=True, fmt='.3f', cmap='coolwarm',
            center=0, linewidths=0.5, annot_kws={'size':11})
plt.title('Heatmap Korelasi Fitur Numerik', fontweight='bold')
plt.tight_layout()
plt.show()

print('Interpretasi:')
print(f'  Korelasi pengalaman-gaji : {corr.loc["pengalaman","gaji"]:.3f} -> hubungan LINEAR KUAT')
print(f'  Korelasi edu-gaji        : {corr.loc["edu","gaji"]:.3f} -> hubungan POSITIF SEDANG')

### Langkah 2 — Preprocessing

In [ ]:
# ── One-Hot Encoding kolom 'kota' ────────────────────────────────
df_enc = pd.get_dummies(df, columns=['kota'], drop_first=True, dtype=int)

print('Kolom setelah One-Hot Encoding:')
print(df_enc.columns.tolist())
df_enc.head()

In [ ]:
# ── Pisahkan Fitur & Target ───────────────────────────────────────
X = df_enc.drop('gaji', axis=1)
y = df_enc['gaji']

print(f'Shape X (fitur)  : {X.shape}')
print(f'Shape y (target) : {y.shape}')
print(f'Fitur            : {X.columns.tolist()}')

# ── Train-Test Split (80:20) ──────────────────────────────────────
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42)

print(f'\nTrain : {X_train.shape[0]} baris (80%)')
print(f'Test  : {X_test.shape[0]} baris (20%)')

# ── StandardScaler (fit HANYA pada training set!) ─────────────────
scaler    = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s  = scaler.transform(X_test)   # hanya transform!

print('\n[OK] Preprocessing selesai!')
print('     Scaler di-fit pada data TRAINING saja -> mencegah Data Leakage')

### Langkah 3 — Latih Model & Tampilkan Koefisien

In [ ]:
# ── Buat & Latih Model ───────────────────────────────────────────
model = LinearRegression()
model.fit(X_train_s, y_train)

print('='*52)
print('HASIL TRAINING MODEL REGRESI LINEAR')
print('='*52)
print(f'beta_0 (intercept) : {model.intercept_:.4f} juta rupiah')
print()

coef_df = pd.DataFrame({
    'Fitur'     : X.columns,
    'Koefisien' : model.coef_.round(4)
}).sort_values('Koefisien', ascending=False).reset_index(drop=True)

print('Koefisien per Fitur (diurutkan terbesar -> terkecil):')
print(coef_df.to_string(index=False))
print()
print('Interpretasi:')
print('  Koefisien positif -> fitur menaikkan prediksi gaji')
print('  Koefisien negatif -> fitur menurunkan prediksi gaji')
print('  Koefisien terbesar -> fitur paling berpengaruh')

In [ ]:
# ── Visualisasi Koefisien ─────────────────────────────────────────
colors = ['#E63946' if c >= 0 else '#457B9D' for c in coef_df['Koefisien']]

plt.figure(figsize=(8, 4))
bars = plt.barh(coef_df['Fitur'], coef_df['Koefisien'],
                color=colors, edgecolor='white', height=0.6)
plt.axvline(0, color='black', linewidth=0.8, linestyle='--')
plt.xlabel('Nilai Koefisien (beta)')
plt.title('Koefisien Model Regresi Linear\n(skala terstandarisasi)', fontweight='bold')

for bar, val in zip(bars, coef_df['Koefisien']):
    xpos = val + 0.05 if val >= 0 else val - 0.05
    ha   = 'left' if val >= 0 else 'right'
    plt.text(xpos, bar.get_y() + bar.get_height()/2,
             f'{val:.3f}', va='center', ha=ha, fontsize=9)

plt.tight_layout()
plt.show()

### Langkah 4 — Evaluasi Model

Tiga metrik utama evaluasi regresi:

$$MAE = \frac{1}{n} \sum |y_i - \hat{y}_i|$$

$$RMSE = \sqrt{\frac{1}{n} \sum (y_i - \hat{y}_i)^2}$$

$$R^2 = 1 - \frac{SS_{res}}{SS_{tot}} = 1 - \frac{\sum(y_i - \hat{y}_i)^2}{\sum(y_i - \bar{y})^2}$$

In [ ]:
# ── Prediksi & Hitung Metrik ──────────────────────────────────────
y_pred = model.predict(X_test_s)

mae  = mean_absolute_error(y_test, y_pred)
mse  = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)
r2   = r2_score(y_test, y_pred)

print('='*52)
print('METRIK EVALUASI MODEL (pada Test Set)')
print('='*52)
print(f'MAE   = {mae:.4f}  juta rupiah')
print(f'MSE   = {mse:.4f}  juta^2')
print(f'RMSE  = {rmse:.4f}  juta rupiah')
print(f'R2    = {r2:.4f}  ({r2*100:.2f}% variasi dapat dijelaskan)')
print(f'\nSelisih RMSE - MAE = {rmse - mae:.4f}')

# Verifikasi via .score()
print(f'Verifikasi R2 via .score(): {model.score(X_test_s, y_test):.4f} [OK]')

In [ ]:
# ── Interpretasi Metrik ───────────────────────────────────────────
gaji_rata = y_test.mean()
print('='*60)
print('INTERPRETASI METRIK EVALUASI')
print('='*60)
print(f'\nGaji rata-rata test set : Rp {gaji_rata:.2f} juta')

print(f'\n[MAE = {mae:.2f} juta]')
print(f'  -> Model rata-rata meleset Rp {mae:.2f} juta dari gaji aktual')
print(f'  -> Relatif thd rata-rata: {mae/gaji_rata*100:.1f}%')

print(f'\n[RMSE = {rmse:.2f} juta]')
if rmse - mae > 0.5:
    print(f'  -> RMSE >> MAE ({rmse-mae:.2f}) -> ada outlier dalam prediksi')
else:
    print(f'  -> RMSE ~ MAE -> kesalahan seragam, tidak ada outlier dominan [OK]')

print(f'\n[R2 = {r2:.4f}]')
if r2 >= 0.8:
    grade = 'SANGAT BAIK'
elif r2 >= 0.6:
    grade = 'BAIK'
elif r2 >= 0.3:
    grade = 'SEDANG'
else:
    grade = 'LEMAH'
print(f'  -> Model menjelaskan {r2*100:.1f}% variasi gaji -> {grade}')

### Langkah 5 — Visualisasi & Interpretasi

In [ ]:
# ── Actual vs Predicted + Residual Plot ──────────────────────────
residuals = y_test - y_pred

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Evaluasi Model Regresi Linear — Prediksi Gaji Karyawan',
             fontsize=13, fontweight='bold')

# Plot 1: Actual vs Predicted
axes[0].scatter(y_test, y_pred, alpha=0.55,
                color='#028090', edgecolors='white', linewidths=0.4, s=40)
lims = [min(y_test.min(), y_pred.min())-1, max(y_test.max(), y_pred.max())+1]
axes[0].plot(lims, lims, 'r--', lw=2, label='Prediksi Sempurna (y=x)')
axes[0].set_xlim(lims); axes[0].set_ylim(lims)
axes[0].set_xlabel('Gaji Aktual (juta Rp)')
axes[0].set_ylabel('Gaji Prediksi (juta Rp)')
axes[0].set_title(f'Actual vs Predicted  (R2 = {r2:.3f})', fontweight='bold')
axes[0].legend()
axes[0].text(lims[0]+0.5, lims[1]-2,
             f'MAE  = {mae:.2f}\nRMSE = {rmse:.2f}\nR2   = {r2:.3f}',
             fontsize=9, va='top',
             bbox=dict(boxstyle='round,pad=0.4', facecolor='white', alpha=0.8))

# Plot 2: Residual Plot
axes[1].scatter(y_pred, residuals, alpha=0.55,
                color='#880E4F', edgecolors='white', linewidths=0.4, s=40)
axes[1].axhline(0, color='gray', linestyle='--', lw=2, label='Residual = 0')
axes[1].set_xlabel('Nilai Prediksi y-hat (juta Rp)')
axes[1].set_ylabel('Residual  y - y-hat  (juta Rp)')
axes[1].set_title('Residual Plot', fontweight='bold')
axes[1].legend()

plt.tight_layout()
plt.savefig('evaluasi_regresi_gaji.png', dpi=150, bbox_inches='tight')
plt.show()
print('Plot tersimpan: evaluasi_regresi_gaji.png')

In [ ]:
# ── Distribusi Residual ───────────────────────────────────────────
plt.figure(figsize=(7, 4))
plt.hist(residuals, bins=25, color='#880E4F', edgecolor='white', alpha=0.8)
plt.axvline(0, color='black', linestyle='--', lw=1.5)
plt.axvline(residuals.mean(), color='orange', lw=2,
            label=f'Mean residual = {residuals.mean():.3f}')
plt.xlabel('Residual (juta Rp)')
plt.ylabel('Frekuensi')
plt.title('Distribusi Residual', fontweight='bold')
plt.legend()
plt.tight_layout()
plt.show()

print(f'Mean residual  : {residuals.mean():.4f}')
print(f'Std  residual  : {residuals.std():.4f}')
print('[OK] Residual mendekati distribusi normal dengan mean ~ 0')

In [ ]:
# ── Cara Membaca Plot ─────────────────────────────────────────────
print('='*60)
print('CARA MEMBACA VISUALISASI EVALUASI')
print('='*60)
print()
print('[Actual vs Predicted Plot]')
print('  WHAT    : Membandingkan nilai aktual (x) vs prediksi (y)')
print('  SO WHAT : Titik mendekati garis merah = model semakin akurat')
print(f'  NOW WHAT: R2 = {r2:.3f} -> model {"BAIK" if r2>=0.6 else "perlu ditingkatkan"}')
print()
print('[Residual Plot]')
print('  WHAT    : Sebaran kesalahan prediksi terhadap y-hat')
print('  SO WHAT : Baik  -> titik acak di sekitar garis y=0')
print('            Buruk -> ada pola (kurva/corong) = asumsi dilanggar')
print('  NOW WHAT: Residual tersebar acak -> asumsi linearitas terpenuhi [OK]')

---
## 📊 4. Perbandingan: Simple vs Multiple Linear Regression

Kita bandingkan model dengan **1 fitur** vs **4 fitur**.

In [ ]:
# ── Simple LR: hanya fitur 'pengalaman' ──────────────────────────
X_s  = df[['pengalaman']]
y_s  = df['gaji']

X_tr_s, X_te_s, y_tr_s, y_te_s = train_test_split(X_s, y_s, test_size=0.2, random_state=42)
sc_s     = StandardScaler()
X_tr_ss  = sc_s.fit_transform(X_tr_s)
X_te_ss  = sc_s.transform(X_te_s)

mdl_s    = LinearRegression()
mdl_s.fit(X_tr_ss, y_tr_s)
y_pred_s = mdl_s.predict(X_te_ss)

mae_s  = mean_absolute_error(y_te_s, y_pred_s)
rmse_s = np.sqrt(mean_squared_error(y_te_s, y_pred_s))
r2_s   = r2_score(y_te_s, y_pred_s)

print('='*58)
print('PERBANDINGAN: Simple LR vs Multiple LR')
print('='*58)
print(f'{"Metrik":<10} {"Simple LR (1 fitur)":>20} {"Multiple LR (4 fitur)":>20}')
print('-'*58)
print(f'{"MAE":<10} {mae_s:>20.4f} {mae:>20.4f}')
print(f'{"RMSE":<10} {rmse_s:>20.4f} {rmse:>20.4f}')
print(f'{"R2":<10} {r2_s:>20.4f} {r2:>20.4f}')
print('='*58)
print(f'\nMultiple LR unggul {r2-r2_s:.4f} poin R2')
print('-> Fitur tambahan (edu, kota) memberikan informasi yang berarti!')

In [ ]:
# ── Visualisasi Garis Regresi Simple LR ──────────────────────────
X_line    = np.linspace(0, 20, 200).reshape(-1, 1)
X_line_sc = sc_s.transform(X_line)
y_line    = mdl_s.predict(X_line_sc)

plt.figure(figsize=(8, 5))
plt.scatter(df['pengalaman'], df['gaji'],
            alpha=0.35, s=20, color='#457B9D', label='Data aktual')
plt.plot(X_line, y_line, color='#E63946', lw=2.5, label='Garis regresi')
plt.xlabel('Pengalaman Kerja (tahun)')
plt.ylabel('Gaji (juta Rp)')
plt.title(
    f'Simple Linear Regression: Pengalaman -> Gaji\n'
    f'b0={mdl_s.intercept_:.2f},  b1={mdl_s.coef_[0]:.2f}  |  R2={r2_s:.3f}',
    fontweight='bold')
plt.legend()
plt.tight_layout()
plt.show()

---
## 📋 5. Ringkasan Metrik Evaluasi Regresi

| Metrik | Formula | Satuan | Keunggulan | Kelemahan | Gunakan Saat |
|--------|---------|--------|------------|-----------|--------------|
| **MAE** | (1/n)Σ\|y−ŷ\| | Sama dengan y | Mudah diinterpretasi; robust outlier | Tidak differentiable di eᵢ=0 | Interpretabilitas prioritas |
| **MSE** | (1/n)Σ(y−ŷ)² | y² | Differentiable; baik untuk optimasi | Satuan dikuadratkan | Internal loss function |
| **RMSE** | √MSE | Sama dengan y | Penalti outlier besar; paling umum | Sensitif outlier ekstrem | Reporting & benchmarking |
| **R²** | 1−SSres/SStot | Tanpa satuan (0–1) | Universal; mudah dibandingkan | Bisa menyesatkan | Membandingkan model |

**Interpretasi R²:**
- `< 0.3` → ❌ Lemah
- `0.3–0.6` → 🟡 Sedang
- `0.6–0.8` → 🟢 Baik
- `> 0.8` → ✅ Sangat Baik *(waspadai overfitting!)*

---
## ✅ Kesimpulan

Pada praktikum Pertemuan 7 ini, telah berhasil dibangun pipeline Machine Learning **end-to-end** menggunakan **Regresi Linear** untuk memprediksi gaji karyawan.

**Ringkasan Pipeline:**
| Tahap | Detail |
|-------|--------|
| Dataset | 300 data sintetis — 3 fitur: pengalaman, edu, kota |
| Preprocessing | One-Hot Encoding (kota), Train-Test Split 80:20, StandardScaler |
| Model | `LinearRegression` scikit-learn (OLS) |
| Evaluasi | MAE, RMSE, R² pada test set |
| Visualisasi | Actual vs Predicted, Residual Plot, Distribusi Residual |

**Pelajaran Utama:**
1. Regresi Linear efektif untuk hubungan **linear** antara X dan y
2. **Multiple LR** unggul dibanding Simple LR karena memanfaatkan lebih banyak informasi
3. Selalu gunakan **minimal 2 metrik** (RMSE + R²) untuk evaluasi komprehensif
4. **Residual Plot** membantu mendeteksi pelanggaran asumsi linearitas

---
**Referensi:**
- Géron, A. (2022). *Hands-On Machine Learning with Scikit-Learn, Keras & TensorFlow* (3rd ed.). O'Reilly Media.
- James, G., et al. (2021). *An Introduction to Statistical Learning* (2nd ed.). Springer.
- scikit-learn Developers. (2024). *LinearRegression*. https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.LinearRegression.html
- Mitchell, T. M. (1997). *Machine Learning*. McGraw-Hill.